In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler


file_path = r"/content/drive/MyDrive/Sent_Analytical-data-final_01082024.csv"
df = pd.read_csv(file_path)

total_rows = len(df)
null_counts = df.isnull().sum()
null_percentage = (null_counts / total_rows) * 100


columns_to_drop = null_percentage[null_percentage > 50].index
df = df.drop(columns=columns_to_drop)

print("Dropped columns:", columns_to_drop.tolist())
print("Remaining columns:", df.columns.tolist())


trace_elements = ['Co (ppm)', 'Ni   (ppm)', 'Cu (ppm)', 'Zn   (ppm)', 'As  (ppm)', 'Se   (ppm)',
                   'Ag (ppm)', 'Sb   (ppm)', 'Te (ppm)', 'Au (ppm)', 'Pb   (ppm)',
                  'Bi   (ppm)',]


for col in trace_elements:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col] = df[col].replace(0, 1e-6)


df_log = df.copy()
df_log[trace_elements] = df_log[trace_elements].applymap(lambda x: np.log(x) if pd.notnull(x) else np.nan)


scaler = MinMaxScaler()
df_scaled = df_log.copy()
df_scaled[trace_elements] = pd.DataFrame(
    scaler.fit_transform(df_log[trace_elements]),
    columns=trace_elements,
    index=df_log.index
)

df_scaled['Deposit type'] = df['Deposit type']


print(f"Preprocessing completed. Shape: {df_scaled.shape}")
df_scaled

In [ ]:

summary_stats = df.groupby('Deposit type').agg(
    {element: ['min', 'max', 'mean', 'std', lambda x: x.std() / x.mean() * 100] for element in df.columns if element != 'Deposit type'}
)

import pandas as pd
import matplotlib.pyplot as plt


fig, ax = plt.subplots(figsize=(20, len(summary_stats) * 0.25))
ax.axis('off')

table = pd.plotting.table(ax, summary_stats.round(3), loc='center', cellLoc='center')

table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.2, 1.2)

plt.show()
plt.savefig('summary_statistics.png', bbox_inches='tight', dpi=300)
plt.close()


In [ ]:
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from scipy.stats import ks_2samp, chi2_contingency
import matplotlib.pyplot as plt
import seaborn as sns


missing_mask = df_scaled[trace_elements].isna()

# --- 3 Imputation Methods ---

def impute_all_methods(df_scaled, trace_elements):
    datasets = {}

    # KNN Imputation
    knn_imputer = KNNImputer(n_neighbors=5)
    df_knn = df_scaled.copy()
    df_knn[trace_elements] = knn_imputer.fit_transform(df_knn[trace_elements])
    datasets['KNN'] = df_knn

    # MICE Imputation
    mice_imputer = IterativeImputer(random_state=0, sample_posterior=True)
    df_mice = df_scaled.copy()
    df_mice[trace_elements] = mice_imputer.fit_transform(df_mice[trace_elements])
    datasets['MICE'] = df_mice

    # Mean Imputation
    mean_imputer = SimpleImputer(strategy='mean')
    df_mean = df_scaled.copy()
    df_mean[trace_elements] = mean_imputer.fit_transform(df_mean[trace_elements])
    datasets['Mean'] = df_mean

    return datasets

imputed_datasets = impute_all_methods(df_scaled, trace_elements)


def evaluate_imputation(imputed_df, original_df, mask, trace_elements):
    ks_scores = {}
    chi2_scores = {}
    for col in trace_elements:
        imputed_vals = imputed_df.loc[mask[col], col]
        observed_vals = original_df.loc[~mask[col], col]


        ks_stat = ks_2samp(imputed_vals, observed_vals).statistic
        ks_scores[col] = ks_stat


        bins = np.histogram_bin_edges(observed_vals.dropna(), bins='auto')
        obs_hist, _ = np.histogram(observed_vals.dropna(), bins=bins)
        imp_hist, _ = np.histogram(imputed_vals.dropna(), bins=bins)
        chi2 = chi2_contingency([obs_hist + 1, imp_hist + 1])[0]
        chi2_scores[col] = chi2

    return ks_scores, chi2_scores

def correlation_diff(original_df, imputed_df, trace_elements):
    corr_orig = original_df[trace_elements].corr()
    corr_imp = imputed_df[trace_elements].corr()
    diff = np.abs(corr_orig - corr_imp)
    return diff.mean().mean()

summary = {}

for method, df_imp in imputed_datasets.items():
    ks, chi2 = evaluate_imputation(df_imp, df_scaled, missing_mask, trace_elements)
    ks_avg = np.mean(list(ks.values()))
    chi2_avg = np.mean(list(chi2.values()))
    corr_dev = correlation_diff(df_scaled.dropna(), df_imp, trace_elements)

    summary[method] = {
        "Avg KS Distance": round(ks_avg, 4),
        "Avg Chi2 Distance": round(chi2_avg, 4),
        "Correlation Deviation": round(corr_dev, 4)
    }


summary_df = pd.DataFrame(summary).T.sort_values(by="Avg KS Distance")
print("📊 Imputation Performance Summary:")
display(summary_df)


plt.figure(figsize=(8, 4))
sns.heatmap(summary_df, annot=True, cmap="Blues", fmt=".4f")
plt.title("Imputation Method Comparison")
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler


df_clr = imputed_datasets['KNN'].copy()
log_values = df_clr[trace_elements]


clr_transformed = log_values.subtract(log_values.mean(axis=1), axis=0)


clr_transformed['Deposit type'] = df['Deposit type']


features = [col for col in clr_transformed.columns if col != 'Deposit type']
X = clr_transformed[features]
y = clr_transformed['Deposit type']


scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# PCA
pca = PCA()
X_pca = pca.fit_transform(X_scaled)


plt.figure(figsize=(8, 5))
plt.plot(range(1, len(pca.explained_variance_ratio_) + 1),
         pca.explained_variance_ratio_, 'o-', linewidth=2)
plt.title('Scree Plot')
plt.xlabel('Principal Component')
plt.ylabel('Variance Explained')
plt.grid(True)
plt.tight_layout()
plt.show()


plt.figure(figsize=(8, 5))
plt.plot(np.cumsum(pca.explained_variance_ratio_), 'o-', linewidth=2)
plt.title('Cumulative Variance Explained by PCA')
plt.xlabel('Number of Principal Components')
plt.ylabel('Cumulative Explained Variance')
plt.grid(True)
plt.axhline(y=0.9, color='r', linestyle='--', label='90% Variance Threshold')
plt.legend()
plt.tight_layout()
plt.show()


pca_df = pd.DataFrame(data=X_pca[:, :2], columns=['PC1', 'PC2'])
pca_df['Deposit type'] = y

plt.figure(figsize=(8, 6))
sns.scatterplot(data=pca_df, x='PC1', y='PC2', hue='Deposit type', palette='tab10')
plt.title('PCA: PC1 vs PC2')
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()


loadings = pd.DataFrame(pca.components_.T,
                        columns=[f'PC{i+1}' for i in range(len(features))],
                        index=features)

plt.figure(figsize=(12, 6))
sns.heatmap(loadings.iloc[:, :5], annot=True, cmap='coolwarm', center=0)
plt.title('PCA Loadings (Top 5 PCs)')
plt.tight_layout()
plt.show()


def biplot(score, coeff, labels=None):
    xs = score[:, 0]
    ys = score[:, 1]
    plt.figure(figsize=(8, 6))
    plt.scatter(xs, ys, alpha=0.6, c=pd.factorize(y)[0], cmap='tab10')

    for i in range(coeff.shape[0]):
        plt.arrow(0, 0,
                  coeff[i, 0]*5, coeff[i, 1]*5,
                  color='black',
                  alpha=0.8,
                  head_width=0.15,
                  head_length=0.2,
                  linewidth=1.8,
                  zorder=5)
        if labels is None:
            plt.text(coeff[i, 0]*5.4, coeff[i, 1]*5.4, f"Var{i+1}", color='darkgreen', fontsize=10)
        else:
            plt.text(coeff[i, 0]*5.4, coeff[i, 1]*5.4, labels[i], color='darkgreen', fontsize=10)

    plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
    plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
    plt.grid(True)
    plt.title('PCA Biplot')
    plt.tight_layout()
    plt.show()


biplot(X_pca, pca.components_.T[:, :2], labels=df.columns.tolist())


explained_variance_ratio = pca.explained_variance_ratio_
print("Explained Variance Ratio for each PC:")
for i, var in enumerate(explained_variance_ratio, start=1):
    print(f"PC{i}: {var:.4f} ({var*100:.2f}%)")


explained_variance = pca.explained_variance_
print("\nExplained Variance (Eigenvalues) for each PC:")
for i, val in enumerate(explained_variance, start=1):
    print(f"PC{i}: {val:.4f}")
cumulative_variance = np.cumsum(explained_variance_ratio)
print("\nCumulative Variance Explained:")
for i, cum_var in enumerate(cumulative_variance, start=1):
    print(f"First {i} PCs: {cum_var:.4f} ({cum_var*100:.2f}%)")


In [ ]:
import pandas as pd
import plotly.express as px



melted_data = df.melt(id_vars='Deposit type', var_name='Element', value_name='Concentration')

# Box Plot

fig = px.box(
    melted_data,
    x='Element',
    y='Concentration',
    color='Deposit type',
    points=False,
    title='Box Plot of Element Concentrations by Deposit Type'
)


fig.update_layout(
    yaxis_title='Concentration (ppm)',
    xaxis_title='Element',
    yaxis_type='log',
    width=1000,
    height=700,
    legend_title='Deposit Type',
    legend=dict(
        orientation='v',
        x=1,
        y=1,
        xanchor='left',
        yanchor='top'
    )
)


fig.show()


In [ ]:
import pandas as pd
import plotly.express as px
from ipywidgets import interact
import scipy.stats as stats
import matplotlib.pyplot as plt

def scatter_plot(x_axis, y_axis, log_x=False, log_y=False):

    fig = px.scatter(
        df, x=x_axis, y=y_axis, color='Deposit type',
        title=f'Scatter plot of {x_axis} vs {y_axis}',
        labels={x_axis: x_axis, y_axis: y_axis}
    )
    fig.update_layout(
        xaxis_type='log' if log_x else 'linear',
        yaxis_type='log' if log_y else 'linear',
        width=700,
        height=500
    )
    file_name = f'scatter_plot_{x_axis}_vs_{y_axis}.html'
    fig.write_html(file_name)

    fig.show()
columns = df.columns.tolist()
interact(scatter_plot, x_axis=columns, y_axis=columns, log_x=True, log_y=True)

scatter_plot


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, BaggingClassifier, StackingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import seaborn as sns
import matplotlib.pyplot as plt


def split_data(df, features, label_col):
    X = df[features]
    y = df[label_col]

    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.99, stratify=y_temp, random_state=42)

    return X_train, X_val, X_test, y_train, y_val, y_test


def train_and_evaluate(X_train, X_val, X_test, y_train, y_val, y_test, label):
    import seaborn as sns
    import matplotlib.pyplot as plt
    from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

    models = {
        'Random Forest': RandomForestClassifier(n_estimators=300, random_state=42),
        'Gradient Boosting': GradientBoostingClassifier(n_estimators=300, random_state=42),
        'Bagging': BaggingClassifier(n_estimators=300, random_state=42),
        'SVM': SVC(kernel='rbf', probability=True, random_state=42),
        'Stacking': StackingClassifier(
            estimators=[('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
                        ('svc', SVC(kernel='rbf', probability=True))],
            final_estimator=LogisticRegression(max_iter=1000), passthrough=False)
    }

    results = {}

    for name, model in models.items():
        print(f"\n Model: {name}")
        model.fit(X_train, y_train)
        preds = model.predict(X_test)

        acc = accuracy_score(y_test, preds)
        report = classification_report(y_test, preds, digits=3)
        cm = confusion_matrix(y_test, preds, labels=np.unique(y_test))


        results[name] = {
            "accuracy": acc,
            "report": report,
            "conf_matrix": cm,
            "model": model
        }


        print(f"Accuracy: {acc:.4f}\n")
        print(" Classification Report:")
        print(report)


        plt.figure(figsize=(6, 5))
        sns.heatmap(pd.DataFrame(cm, index=np.unique(y_test), columns=np.unique(y_test)),
                    annot=True, fmt='d', cmap='YlOrBr', cbar=False)
        plt.title(f"Confusion Matrix: {name}")
        plt.xlabel("Predicted")
        plt.ylabel("Actual")
        plt.tight_layout()
        plt.show()


    print(f"\n Accuracy Comparison for {label} Imputation:")
    accs = {m: results[m]["accuracy"] for m in results}
    fig, ax = plt.subplots(figsize=(10, 5))
    sns.barplot(x=list(accs.keys()), y=list(accs.values()), palette="crest", ax=ax)
    ax.set_ylim(0, 1)
    ax.set_ylabel("Accuracy")
    ax.set_title(f"Test Accuracy by Model ({label} Imputation)")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

    return results


trace_elements = ['Co (ppm)', 'Ni   (ppm)', 'Cu (ppm)', 'Zn   (ppm)', 'As  (ppm)', 'Se   (ppm)',
                   'Ag (ppm)', 'Sb   (ppm)', 'Te (ppm)', 'Au (ppm)', 'Pb   (ppm)',
                  'Bi   (ppm)']
label_col = 'Deposit type'


print(" KNN Imputation Results")
X_train, X_val, X_test, y_train, y_val, y_test = split_data(imputed_datasets['KNN'], trace_elements, label_col)
knn_results = train_and_evaluate(X_train, X_val, X_test, y_train, y_val, y_test, label="KNN")


print("\n MICE Imputation Results")
X_train, X_val, X_test, y_train, y_val, y_test = split_data(imputed_datasets['MICE'], trace_elements, label_col)
mice_results = train_and_evaluate(X_train, X_val, X_test, y_train, y_val, y_test, label="MICE")
